# The Other Repair: Early vs. Late Layer, Matched Within One Model

## What this is, and what it is not

`38`/`40` repaired the *cross-model* comparison (ZymCTRL vs. ProtGPT2, always at layer 12). This
notebook repairs a different, still-open claim: **early-layer-more-sensitive-than-late-layer**,
tested *within* one model (`notes/locked-results.md` §1i/§1k, retracted in §1k-FLAG but never
re-tested at matched strength).

`34-ai4dd-residual-norm-audit.ipynb` confirmed the mechanism directly: the residual stream
accumulates with depth (each layer adds to a running total), so a late layer's hidden state is
naturally larger than an early layer's -- **in every one of the five models tested, 5/5.** Pushing
both layers by the same *absolute* vector norm therefore displaces the early layer's hidden state
by a larger *relative* fraction every time. The reported "early layers are more sensitive" pattern
is exactly what that confound predicts, whether or not early layers are actually more fragile.

## Why p-IgGen, specifically

Four models were tested on this comparison. Only one -- p-IgGen -- reached individual statistical
significance (100.0% vs. 32.0%, non-overlapping CIs, `notes/locked-results.md` §1i). It also has
the largest early/late push asymmetry of any model measured (3.55x, vs. 1.28-1.82x for the other
three, `34`'s audit). A pooled analysis of all four models
(`analysis/meta_analysis.py`) found the entire cross-model significance depends on p-IgGen alone --
drop it and the pooled odds ratio falls from 3.60 (p=6e-8) to 1.72 (p=0.085, not significant).

**So p-IgGen is not just one of four data points -- it is the only one that matters for whether
this claim survives at all.** If p-IgGen's gap shrinks to nothing once genuinely matched, the
early-layer-sensitivity claim has no remaining support anywhere in the project and should be
retracted outright, not softened. If the gap survives, it becomes the first properly-defended
version of the claim, on the model where it was strongest to begin with.

## Design

Reuses `27`'s exact p-IgGen recipe (GPT-NeoX hook path `model.gpt_neox.layers[L]`, the tuple-aware
hook, the single-character `"1"` prompt, layers 1 and 3 of its 4 total). The one change: instead of
normalizing both layers' vectors to the same absolute `REFERENCE_NORM = 583.998`, this notebook
measures p-IgGen's own ‖h‖ at layer 1 *and* layer 3 separately (using the same real-protein-fragment
probe set as `34`/`38`/`40`), and scales each layer's vector to match the **same target α_rel** --
anchored to ProtGPT2's own layer-12 α_rel (0.20), the same anchor `38`/`40` used, so this result is
directly comparable to the rest of the repaired cross-model picture, not just internally consistent.

If layer 1 and layer 3's collapse rates converge once this is done, the original gap was the
confound. If a real gap remains even at matched relative push, that is a genuine, defended
early-vs-late effect -- on the model most likely to show one, tested properly for the first time.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect ~60-90 minutes (p-IgGen is
tiny; ESMFold folding dominates, as in `27`).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.stats import fisher_exact

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"
REFERENCE_NORM = 583.998   # ProtGPT2's own natural v_L norm (03) -- unused for scaling here,
                           # kept only as the historical reference point other notebooks compare to

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())


Setup complete. CUDA available: True


In [2]:
# --- Same common probe set as 34/38/40 -- real UniProt fragments, so the anchor measurement is
#     comparable across every repair notebook in this project, not just internally consistent. ---

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        L = min(frag_len, len(seq))
        start = rng.randint(0, max(1, len(seq) - L + 1))
        probes.append(seq[start:start + L])
    return probes

probe_seqs = build_probe_set(reference_seqs, n_probes=40)
print(f"\nBuilt {len(probe_seqs)} common probe fragments from {len(reference_seqs)} source proteins.")

def measure_resid_norm(model, tokenizer, seqs, layer, hook_path):
    obj = model
    for part in hook_path.split("."):
        obj = getattr(obj, part)
    layer_module = obj[layer]
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
            captured.clear()
            with torch.no_grad():
                model(**inputs)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

print("Probe-measurement function ready.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments from 12 source proteins.
Probe-measurement function ready.


In [3]:
# --- Step 1: the anchor, ProtGPT2's own layer-12 alpha_rel -- SAME anchor as 38/40, so this
#     result sits on the same scale as the ZymCTRL repair rather than being its own island. ---

print(f"Loading ProtGPT2 on {device} to measure the anchor...")
anchor_tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
anchor_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
anchor_model.eval()

h_protgpt2_l12 = measure_resid_norm(anchor_model, anchor_tokenizer, probe_seqs, 12,
                                    hook_path="transformer.h")
anchor_alpha_rel = REFERENCE_NORM / h_protgpt2_l12

print(f"ProtGPT2 layer 12 mean residual-stream norm (this run's probes): {h_protgpt2_l12:.2f}")
print(f"anchor_alpha_rel = {anchor_alpha_rel:.4f}")
print(f"(38 measured 0.2000, 40 should measure something close -- consistency across runs is the")
print(f" check that this anchor is a stable quantity, not a fluke of one measurement.)")

del anchor_model, anchor_tokenizer
clear_gpu()
print("\nProtGPT2 freed from GPU.")


Loading ProtGPT2 on cuda to measure the anchor...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtGPT2 layer 12 mean residual-stream norm (this run's probes): 2919.92
anchor_alpha_rel = 0.2000
(38 measured 0.2000, 40 should measure something close -- consistency across runs is the
 check that this anchor is a stable quantity, not a fluke of one measurement.)

ProtGPT2 freed from GPU.


In [4]:
# --- Scoring + ESMFold, unchanged from 24/27, plus the length-aware fix from 39/40. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and length-aware ESMFold evaluator ready.")


Scoring functions and length-aware ESMFold evaluator ready.


In [5]:
# --- p-IgGen-specific loading, prompt, and output-cleaning -- identical to 27. ---

print(f"Loading p-IgGen on {device}...")
tokenizer = AutoTokenizer.from_pretrained("opig/p-IgGen")
plm_model = AutoModelForCausalLM.from_pretrained("opig/p-IgGen").to(device)
plm_model.eval()

N_LAYERS = plm_model.config.num_hidden_layers
LAYER_EARLY = 1
LAYER_LATE = N_LAYERS - 1
print(f"p-IgGen has {N_LAYERS} hidden layers -- using layer {LAYER_EARLY} (~1/4 depth) and layer "
      f"{LAYER_LATE} (final layer) as the two test points, same as 27.")

PROMPT_CHAR = "1"
EOS_CHAR = "2"
PROMPT_TOKEN_ID = tokenizer.encode(PROMPT_CHAR)[0]
EOS_TOKEN_ID = tokenizer.encode(EOS_CHAR)[0]
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else EOS_TOKEN_ID

def clean_piggen_output(decoded_text):
    text = decoded_text
    if text.startswith(PROMPT_CHAR):
        text = text[1:]
    if EOS_CHAR in text:
        text = text.split(EOS_CHAR)[0]
    return text

def generate_natural(tokenizer, model, n_sequences, max_len=50, seed=0):
    torch.manual_seed(seed)
    records = []
    for i in range(n_sequences):
        inputs = tokenizer(PROMPT_CHAR, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, top_p=0.95,
                eos_token_id=EOS_TOKEN_ID, pad_token_id=PAD_TOKEN_ID
            )
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_piggen_output(raw_decoded)
        records.append({"raw_decoded": raw_decoded, "sequence": seq, "gen_only": seq})
    clear_gpu()
    return records

N_CANDIDATES = 200
print(f"=== Generating N={N_CANDIDATES} natural candidate sequences ===")
candidate_records = generate_natural(tokenizer, plm_model, N_CANDIDATES, max_len=50, seed=505)

print("=== Freeing p-IgGen while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = StructuralEvaluatorPTM()
print("Folding and scoring the candidate pool...")
candidate_records = fold_records_ptm(candidate_records, evaluator)
del evaluator
clear_gpu()

valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
print(f"\n{len(valid_candidates)}/{len(candidate_records)} candidates folded successfully.")
print(f"Mean pLDDT: {np.mean([r['plddt'] for r in valid_candidates]):.2f}, "
      f"natural collapse rate: {np.mean([r['collapse'] for r in valid_candidates]):.1%}")
print(f"(27 recorded 0.0% natural collapse, mean pLDDT 74.36 -- p-IgGen's output is naturally")
print(f" very reliable; close values here confirm the pool is comparable.)")


Loading p-IgGen on cuda...


config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding and scoring the candidate pool...

200/200 candidates folded successfully.
Mean pLDDT: 74.36, natural collapse rate: 0.0%
(27 recorded 0.0% natural collapse, mean pLDDT 74.36 -- p-IgGen's output is naturally
 very reliable; close values here confirm the pool is comparable.)


In [6]:
# --- Utility-matching, unchanged. NOTE: 27 found this pool's D+/D- utility gap was already
#     tiny before matching (n=60/60 kept), since p-IgGen's natural output rarely fails. Expect the
#     same here. ---

QUANTILE = 0.30
UTILITY_TOLERANCE = 0.05

sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(sorted_by_rep) * QUANTILE))
d_minus_raw = sorted_by_rep[:n_side]
d_plus_raw = sorted_by_rep[-n_side:]

def utility_match(pool_a, pool_b, tolerance, max_iters=200):
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r["utility_score"] for r in a])
        mean_b = np.mean([r["utility_score"] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r["utility_score"]); a.pop(0)
        else:
            b.sort(key=lambda r: r["utility_score"]); b.pop(0)
    return a, b

d_plus, d_minus = utility_match(d_plus_raw, d_minus_raw, UTILITY_TOLERANCE)
print(f"After utility-matching: D+ n={len(d_plus)}, D- n={len(d_minus)}, "
      f"utility gap {abs(np.mean([r['utility_score'] for r in d_plus]) - np.mean([r['utility_score'] for r in d_minus])):.3f}")


After utility-matching: D+ n=60, D- n=60, utility gap 0.002


In [7]:
# --- Step 2+3: build BOTH layers' vectors AND measure p-IgGen's own ‖h‖ at BOTH layers
#     separately, using the same probe fragments as the anchor. This is the actual fix -- 27
#     normalized both layers to the same absolute 583.998; this notebook gives each layer its
#     own matched norm = anchor_alpha_rel * h(that layer). ---

print(f"Reloading p-IgGen on {device} to extract activations and measure ‖h‖...")
plm_model = AutoModelForCausalLM.from_pretrained("opig/p-IgGen").to(device)
plm_model.eval()

def get_mean_activation(model, tokenizer, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

d_plus_seqs = [r["sequence"] for r in d_plus]
d_minus_seqs = [r["sequence"] for r in d_minus]

steering_vectors = {}
h_piggen = {}
matched_norms = {}
for layer in [LAYER_EARLY, LAYER_LATE]:
    pos_acts = get_mean_activation(plm_model, tokenizer, d_plus_seqs, layer)
    neg_acts = get_mean_activation(plm_model, tokenizer, d_minus_seqs, layer)
    v_raw = pos_acts.mean(dim=0) - neg_acts.mean(dim=0)
    raw_norm = v_raw.norm().item()

    h = measure_resid_norm(plm_model, tokenizer, probe_seqs, layer, hook_path="gpt_neox.layers")
    h_piggen[layer] = h
    matched_norm = anchor_alpha_rel * h
    matched_norms[layer] = matched_norm

    v_matched = v_raw * (matched_norm / raw_norm)
    steering_vectors[layer] = v_matched.to(device)

    print(f"Layer {layer}: raw utility-matched v_L norm = {raw_norm:.4f}")
    print(f"  p-IgGen ‖h‖ at this layer (this run's probes): {h:.2f}")
    print(f"  matched norm for anchor_alpha_rel={anchor_alpha_rel:.4f}: {matched_norm:.4f}")
    print(f"  (27 used a fixed {REFERENCE_NORM:.3f} here regardless of layer -- "
          f"{matched_norm / REFERENCE_NORM:.4f}x that value)")

within_model_asymmetry = matched_norms[LAYER_LATE] / matched_norms[LAYER_EARLY] * \
                         h_piggen[LAYER_EARLY] / h_piggen[LAYER_LATE]
print(f"\nSanity check -- matched alpha_rel should now be IDENTICAL at both layers:")
print(f"  layer {LAYER_EARLY}: {matched_norms[LAYER_EARLY] / h_piggen[LAYER_EARLY]:.4f}")
print(f"  layer {LAYER_LATE}: {matched_norms[LAYER_LATE] / h_piggen[LAYER_LATE]:.4f}")
print(f"  (both should equal anchor_alpha_rel={anchor_alpha_rel:.4f} -- if they don't, something")
print(f"   is wrong with the scaling above)")


Reloading p-IgGen on cuda to extract activations and measure ‖h‖...


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

Layer 1: raw utility-matched v_L norm = 0.5670
  p-IgGen ‖h‖ at this layer (this run's probes): 6.13
  matched norm for anchor_alpha_rel=0.2000: 1.2257
  (27 used a fixed 583.998 here regardless of layer -- 0.0021x that value)
Layer 3: raw utility-matched v_L norm = 1.8319
  p-IgGen ‖h‖ at this layer (this run's probes): 21.41
  matched norm for anchor_alpha_rel=0.2000: 4.2812
  (27 used a fixed 583.998 here regardless of layer -- 0.0073x that value)

Sanity check -- matched alpha_rel should now be IDENTICAL at both layers:
  layer 1: 0.2000
  layer 3: 0.2000
  (both should equal anchor_alpha_rel=0.2000 -- if they don't, something
   is wrong with the scaling above)


In [8]:
# --- Steering: GPT-NeoX hook path and tuple-aware hook, identical to 27. Only vector SCALE
#     differs. ---

def generate_with_vector_steering(model, tokenizer, target_layer, steering_vector, n_sequences, max_len=50, seed=0):
    torch.manual_seed(seed)
    model.eval()
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        if isinstance(out, tuple):
            return (out[0] + v,) + out[1:]
        return out + v

    records = []
    for i in range(n_sequences):
        handle = model.gpt_neox.layers[target_layer].register_forward_hook(hook)
        inputs = tokenizer(PROMPT_CHAR, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, top_p=0.95,
                eos_token_id=EOS_TOKEN_ID, pad_token_id=PAD_TOKEN_ID
            )
        handle.remove()
        raw_decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        seq = clean_piggen_output(raw_decoded)
        records.append({"sequence": seq, "entropy": calculate_entropy(seq)})
    clear_gpu()
    return records

N_PER_CONDITION = 50
conditions = {}

print("=== CONTROL (unsteered) ===")
conditions["CONTROL"] = generate_with_vector_steering(
    plm_model, tokenizer, LAYER_EARLY, None, N_PER_CONDITION, seed=111)

print(f"=== L{LAYER_EARLY}_MATCHED_1x (alpha_rel-matched to ProtGPT2's anchor) ===")
conditions[f"L{LAYER_EARLY}_MATCHED_1x"] = generate_with_vector_steering(
    plm_model, tokenizer, LAYER_EARLY, steering_vectors[LAYER_EARLY] * 1.0, N_PER_CONDITION, seed=222)

print(f"=== L{LAYER_EARLY}_MATCHED_2x ===")
conditions[f"L{LAYER_EARLY}_MATCHED_2x"] = generate_with_vector_steering(
    plm_model, tokenizer, LAYER_EARLY, steering_vectors[LAYER_EARLY] * 2.0, N_PER_CONDITION, seed=333)

print(f"=== L{LAYER_LATE}_MATCHED_1x ===")
conditions[f"L{LAYER_LATE}_MATCHED_1x"] = generate_with_vector_steering(
    plm_model, tokenizer, LAYER_LATE, steering_vectors[LAYER_LATE] * 1.0, N_PER_CONDITION, seed=444)

print(f"=== L{LAYER_LATE}_MATCHED_2x ===")
conditions[f"L{LAYER_LATE}_MATCHED_2x"] = generate_with_vector_steering(
    plm_model, tokenizer, LAYER_LATE, steering_vectors[LAYER_LATE] * 2.0, N_PER_CONDITION, seed=555)

for name, recs in conditions.items():
    print(f"{name}: {len(recs)} sequences generated")

print("=== Freeing p-IgGen from GPU ===")
del plm_model
clear_gpu()


=== CONTROL (unsteered) ===
=== L1_MATCHED_1x (alpha_rel-matched to ProtGPT2's anchor) ===
=== L1_MATCHED_2x ===
=== L3_MATCHED_1x ===
=== L3_MATCHED_2x ===
CONTROL: 50 sequences generated
L1_MATCHED_1x: 50 sequences generated
L1_MATCHED_2x: 50 sequences generated
L3_MATCHED_1x: 50 sequences generated
L3_MATCHED_2x: 50 sequences generated
=== Freeing p-IgGen from GPU ===


In [9]:
# --- Fold with fold-success tracking. ---
evaluator2 = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    conditions[name] = fold_records_ptm(conditions[name], evaluator2)
del evaluator2
clear_gpu()

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

print(f"\n{'Condition':20s} {'N':>4s} {'FoldOK':>7s} {'Entropy':>9s} {'pLDDT':>8s} {'Collapse%':>10s}")
print("-" * 72)
summary = {}
for name, recs in conditions.items():
    n = len(recs)
    n_ok = sum(1 for r in recs if r["fold_ok"])
    k = int(np.sum([r["collapse"] for r in recs]))
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0]
    summary[name] = {"k": k, "n": n, "rate": k / n, "ci": wilson_ci(k, n), "fold_ok": n_ok}
    print(f"{name:20s} {n:4d} {n_ok:3d}/{n:<3d} {np.mean([r['entropy'] for r in recs]):9.3f} "
          f"{(np.mean(plddts) if plddts else 0.0):8.2f} {k / n * 100:9.1f}%")


Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding L1_MATCHED_1x...
Folding L1_MATCHED_2x...
Folding L3_MATCHED_1x...
Folding L3_MATCHED_2x...

Condition               N  FoldOK   Entropy    pLDDT  Collapse%
------------------------------------------------------------------------
CONTROL                50  50/50      3.722    74.21       0.0%
L1_MATCHED_1x          50  50/50      3.718    74.23       0.0%
L1_MATCHED_2x          50  50/50      3.722    74.00       0.0%
L3_MATCHED_1x          50  50/50      3.731    74.37       0.0%
L3_MATCHED_2x          50  50/50      3.751    74.51       0.0%


In [10]:
# --- The actual comparison this notebook exists to make. ---

print("=" * 100)
print("THE REPAIRED COMPARISON: early (layer 1) vs late (layer 3) at genuinely matched relative push")
print("=" * 100)
print(f"anchor_alpha_rel: {anchor_alpha_rel:.4f}")
print()
print(f"{'Condition':20s} {'collapsed':>11s} {'rate':>8s} {'95% CI':>20s}")
print("-" * 68)
for name, s in summary.items():
    lo, hi = s["ci"]
    ci_str = "[{:.1%}, {:.1%}]".format(lo, hi)
    print(f"{name:20s} {s['k']:5d}/{s['n']:<5d} {s['rate']:7.1%} {ci_str:>20s}")

print()
print("Read against 27's ORIGINAL (absolute-norm, mismatched) result (locked-results.md SS1i):")
print("  CONTROL 0.0% -> L1_FAIR_1x 100.0% -> L1_FAIR_2x 100.0% -> L3_FAIR_1x 32.0% -> L3_FAIR_2x 34.0%")
print(f"  (that L1 '1x' was actually alpha_rel = 584/{h_piggen[LAYER_EARLY]:.2f} = "
      f"{REFERENCE_NORM / h_piggen[LAYER_EARLY]:.2f} -- "
      f"{(REFERENCE_NORM / h_piggen[LAYER_EARLY]) / anchor_alpha_rel:.1f}x harder than this run's matched 1x)")
print(f"  (that L3 '1x' was actually alpha_rel = 584/{h_piggen[LAYER_LATE]:.2f} = "
      f"{REFERENCE_NORM / h_piggen[LAYER_LATE]:.2f} -- "
      f"{(REFERENCE_NORM / h_piggen[LAYER_LATE]) / anchor_alpha_rel:.1f}x harder than this run's matched 1x)")

ctrl = summary["CONTROL"]
early1 = summary[f"L{LAYER_EARLY}_MATCHED_1x"]
early2 = summary[f"L{LAYER_EARLY}_MATCHED_2x"]
late1 = summary[f"L{LAYER_LATE}_MATCHED_1x"]
late2 = summary[f"L{LAYER_LATE}_MATCHED_2x"]

print()
print(f"This run (matched to ProtGPT2's own alpha_rel):")
print(f"  CONTROL {ctrl['rate']:.1%} -> L{LAYER_EARLY}_1x {early1['rate']:.1%} -> "
      f"L{LAYER_EARLY}_2x {early2['rate']:.1%} -> L{LAYER_LATE}_1x {late1['rate']:.1%} -> "
      f"L{LAYER_LATE}_2x {late2['rate']:.1%}")

print()
print("=" * 100)
print("VERDICT: does the early > late gap survive matching?")
print("=" * 100)

_, p_1x_gap = fisher_exact([[early1["k"], early1["n"] - early1["k"]],
                            [late1["k"], late1["n"] - late1["k"]]])
_, p_2x_gap = fisher_exact([[early2["k"], early2["n"] - early2["k"]],
                            [late2["k"], late2["n"] - late2["k"]]])

print(f"Layer {LAYER_EARLY} vs Layer {LAYER_LATE} at matched 1x: "
      f"{early1['rate']:.1%} vs {late1['rate']:.1%}, Fisher p = {p_1x_gap:.4f}")
print(f"Layer {LAYER_EARLY} vs Layer {LAYER_LATE} at matched 2x: "
      f"{early2['rate']:.1%} vs {late2['rate']:.1%}, Fisher p = {p_2x_gap:.4f}")
print()

gap_survives_1x = p_1x_gap < 0.05 and early1["rate"] > late1["rate"]
gap_survives_2x = p_2x_gap < 0.05 and early2["rate"] > late2["rate"]

if gap_survives_1x or gap_survives_2x:
    print("  ==> THE GAP SURVIVES matching, at least at one dose. This means the original")
    print("      early > late finding was NOT purely the confound -- there is a real, defended")
    print("      layer-sensitivity effect on p-IgGen even at genuinely matched relative push.")
    print("      This is now the first properly-controlled version of that claim anywhere in the")
    print("      project. Update locked-results.md SS1i/SS1k to promote this as the real evidence,")
    print("      and context-and-decisions.md SS10 to un-retract the claim FOR THIS MODEL")
    print("      specifically -- do not generalize back to ProtGPT2/ZymCTRL/RITA without testing")
    print("      them the same way; each showed a much smaller original gap even before matching.")
else:
    print("  ==> THE GAP DOES NOT SURVIVE matching. Once layer 1 and layer 3 are pushed by the")
    print("      same fraction of their own hidden state, they behave statistically alike.")
    print("      This confirms the original 100.0% vs 32.0% gap was the confound, not a real")
    print("      early-vs-late effect. Combined with the pooled-analysis finding that p-IgGen was")
    print("      the ENTIRE basis for the cross-model 'early > late' claim, this means the claim")
    print("      has no remaining support anywhere in the project. Retract it fully in")
    print("      locked-results.md SS1i/SS1k and context-and-decisions.md SS10 -- not as a hedge,")
    print("      as a closed, resolved question. This is a real, citable negative result in its")
    print("      own right: 'the apparent layer-sensitivity pattern was a measurement artifact,")
    print("      confirmed by testing the strongest individual case directly.'")


THE REPAIRED COMPARISON: early (layer 1) vs late (layer 3) at genuinely matched relative push
anchor_alpha_rel: 0.2000

Condition              collapsed     rate               95% CI
--------------------------------------------------------------------
CONTROL                  0/50       0.0%         [0.0%, 7.1%]
L1_MATCHED_1x            0/50       0.0%         [0.0%, 7.1%]
L1_MATCHED_2x            0/50       0.0%         [0.0%, 7.1%]
L3_MATCHED_1x            0/50       0.0%         [0.0%, 7.1%]
L3_MATCHED_2x            0/50       0.0%         [0.0%, 7.1%]

Read against 27's ORIGINAL (absolute-norm, mismatched) result (locked-results.md SS1i):
  CONTROL 0.0% -> L1_FAIR_1x 100.0% -> L1_FAIR_2x 100.0% -> L3_FAIR_1x 32.0% -> L3_FAIR_2x 34.0%
  (that L1 '1x' was actually alpha_rel = 584/6.13 = 95.29 -- 476.5x harder than this run's matched 1x)
  (that L3 '1x' was actually alpha_rel = 584/21.41 = 27.28 -- 136.4x harder than this run's matched 1x)

This run (matched to ProtGPT2's own alpha_re

In [11]:
# --- Persist. ---
rows = []
for name, recs in conditions.items():
    layer = LAYER_EARLY if f"L{LAYER_EARLY}" in name else (LAYER_LATE if f"L{LAYER_LATE}" in name else None)
    mult = 2.0 if name.endswith("2x") else (1.0 if name.endswith("1x") else 0.0)
    a_rel = 0.0 if layer is None else mult * matched_norms[layer] / h_piggen[layer]
    for i, r in enumerate(recs):
        rows.append({
            "condition": name, "idx": i, "layer": layer, "multiplier": mult, "alpha_rel": a_rel,
            "sequence": r["sequence"], "gen_length": len(r["sequence"]),
            "usable_length": sum(1 for a in r["sequence"] if a in VALID_AA),
            "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"],
            "fold_ok": r["fold_ok"], "collapse": r["collapse"],
        })
pd.DataFrame(rows).to_csv("piggen_layer_matched_sequences.csv", index=False)

pd.DataFrame([{
    "condition": n, "collapsed": s["k"], "n": s["n"], "collapse_rate": s["rate"],
    "ci_lo": s["ci"][0], "ci_hi": s["ci"][1], "fold_ok": s["fold_ok"],
} for n, s in summary.items()]).to_csv("piggen_layer_matched_summary.csv", index=False)

pd.DataFrame([{
    "anchor_alpha_rel": anchor_alpha_rel, "h_protgpt2_l12": h_protgpt2_l12,
    "h_piggen_early": h_piggen[LAYER_EARLY], "h_piggen_late": h_piggen[LAYER_LATE],
    "matched_norm_early": matched_norms[LAYER_EARLY], "matched_norm_late": matched_norms[LAYER_LATE],
    "reference_norm_used_by_27": REFERENCE_NORM,
}]).to_csv("piggen_layer_matched_calibration.csv", index=False)

print("Saved:")
print("  piggen_layer_matched_sequences.csv")
print("  piggen_layer_matched_summary.csv")
print("  piggen_layer_matched_calibration.csv")
print()
print("Update notes/locked-results.md SS1i/SS1k and notes/context-and-decisions.md SS10 with this")
print("run's verdict -- it resolves whether the early-vs-late-layer claim has ANY remaining")
print("support anywhere in the project.")


Saved:
  piggen_layer_matched_sequences.csv
  piggen_layer_matched_summary.csv
  piggen_layer_matched_calibration.csv

Update notes/locked-results.md SS1i/SS1k and notes/context-and-decisions.md SS10 with this
run's verdict -- it resolves whether the early-vs-late-layer claim has ANY remaining
support anywhere in the project.
